In [0]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor


INPUT_FILE = "V4 Usage Training.csv"
OUTPUT_FOLDER = Path("forecast_results")

# Rolling backtest settings
VALIDATION_START = "2026-01-01"
FORECAST_MONTHS = 6

TARGET_COLUMN = "Monthly Inventory Issues"
PART_COLUMN = "fpartno"
DATE_COLUMN = "Date"

# Version 5 commitment feature
COMMITS_COLUMN = "Avg Commits/Month"

LAGS = [1, 2, 3, 6, 12]

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True,
)

print("Version 5 settings loaded.")

Version 5 settings loaded.


In [0]:
data = pd.read_csv(INPUT_FILE)

required_columns = {
    PART_COLUMN,
    DATE_COLUMN,
    TARGET_COLUMN,
    COMMITS_COLUMN,
}

missing_columns = required_columns.difference(
    data.columns
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: "
        f"{sorted(missing_columns)}"
    )

data[DATE_COLUMN] = pd.to_datetime(
    data[DATE_COLUMN],
    errors="raise",
)

# Keep only data from 2023 onward
data = data[
    data[DATE_COLUMN] >= pd.Timestamp("2023-01-01")
].copy()

data[TARGET_COLUMN] = pd.to_numeric(
    data[TARGET_COLUMN],
    errors="raise",
)

# Convert commitments to numeric
data[COMMITS_COLUMN] = pd.to_numeric(
    data[COMMITS_COLUMN],
    errors="coerce",
)

# Only keep this if blank means no commitments
data[COMMITS_COLUMN] = (
    data[COMMITS_COLUMN]
    .fillna(0)
)

data = (
    data
    .dropna(
        subset=[
            PART_COLUMN,
            DATE_COLUMN,
            TARGET_COLUMN,
        ]
    )
    .sort_values(
        [
            PART_COLUMN,
            DATE_COLUMN,
        ]
    )
    .reset_index(drop=True)
)

duplicates = data.duplicated(
    [
        PART_COLUMN,
        DATE_COLUMN,
    ],
    keep=False,
)

if duplicates.any():

    duplicate_rows = data.loc[
        duplicates,
        [
            PART_COLUMN,
            DATE_COLUMN,
        ],
    ]

    raise ValueError(
        "Duplicate part/month rows were found:\n"
        f"{duplicate_rows.head(20)}"
    )

print(f"Rows loaded: {len(data):,}")

print(
    f"Unique parts: "
    f"{data[PART_COLUMN].nunique():,}"
)

print(
    f"First date: "
    f"{data[DATE_COLUMN].min()}"
)

print(
    f"Last date: "
    f"{data[DATE_COLUMN].max()}"
)

print(
    f"Average commitments: "
    f"{data[COMMITS_COLUMN].mean():.2f}"
)

print(
    f"Rows with commitments: "
    f"{(data[COMMITS_COLUMN] > 0).sum():,}"
)

Rows loaded: 748
Unique parts: 17
First date: 2023-01-01 00:00:00
Last date: 2026-08-01 00:00:00
Average commitments: 27.35
Rows with commitments: 655


In [0]:
def create_training_features(historical_data):

    feature_data = historical_data.copy()

    # =====================================================
    # DATE / TIME FEATURES
    # =====================================================

    feature_data["month"] = (
        feature_data[DATE_COLUMN].dt.month
    )

    feature_data["year"] = (
        feature_data[DATE_COLUMN].dt.year
    )

    feature_data["quarter"] = (
        feature_data[DATE_COLUMN].dt.quarter
    )

    feature_data["time_idx"] = (
        np.arange(len(feature_data))
    )


    # =====================================================
    # USAGE LAG FEATURES
    # =====================================================

    for lag in LAGS:

        feature_data[f"lag_{lag}"] = (
            feature_data[TARGET_COLUMN].shift(lag)
        )


    # Shift usage so current usage can never
    # be used to predict itself.
    prior_usage = (
        feature_data[TARGET_COLUMN].shift(1)
    )


    # =====================================================
    # ROLLING USAGE AVERAGES
    # =====================================================

    feature_data["rolling_mean_3"] = (
        prior_usage.rolling(3).mean()
    )

    feature_data["rolling_mean_6"] = (
        prior_usage.rolling(6).mean()
    )

    feature_data["rolling_mean_12"] = (
        prior_usage.rolling(12).mean()
    )


    # =====================================================
    # ROLLING USAGE MEDIANS
    # =====================================================

    feature_data["rolling_median_3"] = (
        prior_usage.rolling(3).median()
    )

    feature_data["rolling_median_6"] = (
        prior_usage.rolling(6).median()
    )

    feature_data["rolling_median_12"] = (
        prior_usage.rolling(12).median()
    )


    # =====================================================
    # ANNUAL USAGE
    # =====================================================

    feature_data["rolling_total_12"] = (
        prior_usage.rolling(12).sum()
    )


    # =====================================================
    # USAGE VARIABILITY
    # =====================================================

    feature_data["rolling_std_3"] = (
        prior_usage.rolling(3).std()
    )

    feature_data["rolling_std_6"] = (
        prior_usage.rolling(6).std()
    )

    feature_data["rolling_std_12"] = (
        prior_usage.rolling(12).std()
    )


    # =====================================================
    # USAGE RANGE
    # =====================================================

    feature_data["rolling_min_12"] = (
        prior_usage.rolling(12).min()
    )

    feature_data["rolling_max_12"] = (
        prior_usage.rolling(12).max()
    )


    # =====================================================
    # USAGE TRENDS
    # =====================================================

    feature_data["trend_3"] = (
        feature_data["lag_1"]
        - feature_data["lag_3"]
    )

    feature_data["trend_6"] = (
        feature_data["lag_1"]
        - feature_data["lag_6"]
    )


    # =====================================================
    # ZERO-USAGE BEHAVIOR
    # =====================================================

    feature_data[
        "zero_month_percentage_12"
    ] = (
        prior_usage
        .rolling(12)
        .apply(
            lambda values: (
                values == 0
            ).mean(),
            raw=True,
        )
    )


    # =====================================================
    # COEFFICIENT OF VARIATION
    # =====================================================

    feature_data[
        "coefficient_variation_12"
    ] = (
        feature_data["rolling_std_12"]
        /
        feature_data[
            "rolling_mean_12"
        ].replace(
            0,
            np.nan,
        )
    )


    # =====================================================
    # RECENT VS ANNUAL USAGE
    # =====================================================

    feature_data["recent_vs_annual"] = (
        feature_data["rolling_mean_3"]
        -
        feature_data["rolling_mean_12"]
    )


    # =====================================================
    # COMMITMENT FEATURES
    # =====================================================

    # Shift commitments so the model does not see
    # the commitment value from the month being predicted.
    prior_commits = (
        feature_data[COMMITS_COLUMN].shift(1)
    )


    # Commitment lags
    feature_data["commits_lag_1"] = (
        feature_data[COMMITS_COLUMN].shift(1)
    )

    feature_data["commits_lag_2"] = (
        feature_data[COMMITS_COLUMN].shift(2)
    )

    feature_data["commits_lag_3"] = (
        feature_data[COMMITS_COLUMN].shift(3)
    )

    feature_data["commits_lag_6"] = (
        feature_data[COMMITS_COLUMN].shift(6)
    )


    # Rolling commitment averages
    feature_data["commits_rolling_mean_3"] = (
        prior_commits.rolling(3).mean()
    )

    feature_data["commits_rolling_mean_6"] = (
        prior_commits.rolling(6).mean()
    )

    feature_data["commits_rolling_mean_12"] = (
        prior_commits.rolling(12).mean()
    )


    # Rolling commitment totals
    feature_data["commits_rolling_sum_3"] = (
        prior_commits.rolling(3).sum()
    )

    feature_data["commits_rolling_sum_6"] = (
        prior_commits.rolling(6).sum()
    )


    # Compare recent commitment pressure with
    # the longer-term commitment level.
    feature_data[
        "commits_recent_vs_annual"
    ] = (
        feature_data[
            "commits_rolling_mean_3"
        ]
        -
        feature_data[
            "commits_rolling_mean_12"
        ]
    )


    # =====================================================
    # FEATURE LIST
    # =====================================================

    feature_columns = [

        # Date / time
        "month",
        "year",
        "quarter",
        "time_idx",

        # Usage lags
        "lag_1",
        "lag_2",
        "lag_3",
        "lag_6",
        "lag_12",

        # Usage averages
        "rolling_mean_3",
        "rolling_mean_6",
        "rolling_mean_12",

        # Usage medians
        "rolling_median_3",
        "rolling_median_6",
        "rolling_median_12",

        # Annual usage
        "rolling_total_12",

        # Usage variability
        "rolling_std_3",
        "rolling_std_6",
        "rolling_std_12",

        # Usage range
        "rolling_min_12",
        "rolling_max_12",

        # Demand behavior
        "zero_month_percentage_12",
        "coefficient_variation_12",
        "recent_vs_annual",

        # Usage trends
        "trend_3",
        "trend_6",

        # Commitment features
        "commits_lag_1",
        "commits_lag_2",
        "commits_lag_3",
        "commits_lag_6",

        "commits_rolling_mean_3",
        "commits_rolling_mean_6",
        "commits_rolling_mean_12",

        "commits_rolling_sum_3",
        "commits_rolling_sum_6",

        "commits_recent_vs_annual",
    ]


    training_rows = (
        feature_data
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna(
            subset=feature_columns
        )
        .reset_index(drop=True)
    )


    return (
        training_rows,
        feature_columns,
    )


print(
    "Version 5 commitment feature function created."
)

Version 5 commitment feature function created.


In [0]:
def train_model(training_rows, feature_columns):

    model = LGBMRegressor(
        objective="poisson",
        n_estimators=250,
        learning_rate=0.05,
        max_depth=5,
        min_child_samples=10,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1,
    )

    model.fit(
        training_rows[feature_columns],
        training_rows[TARGET_COLUMN],
    )

    return model


print("Training function created.")

Training function created.


In [0]:
def forecast_future_months(
    model,
    historical_data,
    feature_columns,
    forecast_months,
    part_number,
):

    forecast_history = (
        historical_data[
            [
                DATE_COLUMN,
                TARGET_COLUMN,
                COMMITS_COLUMN,
            ]
        ]
        .copy()
        .sort_values(DATE_COLUMN)
        .reset_index(drop=True)
    )

    predictions = []


    for _ in range(forecast_months):

        next_date = (
            forecast_history[DATE_COLUMN].max()
            + pd.DateOffset(months=1)
        )

        recent_usage = (
            forecast_history[TARGET_COLUMN]
        )

        recent_commits = (
            forecast_history[COMMITS_COLUMN]
        )


        # =====================================================
        # DATE / TIME
        # =====================================================

        future_row = {
            "month": next_date.month,
            "year": next_date.year,
            "quarter": next_date.quarter,
            "time_idx": len(forecast_history),
        }


        # =====================================================
        # USAGE FEATURES
        # =====================================================

        for lag in LAGS:

            future_row[f"lag_{lag}"] = (
                recent_usage.iloc[-lag]
            )


        last_3 = recent_usage.iloc[-3:]
        last_6 = recent_usage.iloc[-6:]
        last_12 = recent_usage.iloc[-12:]


        # Rolling averages
        future_row["rolling_mean_3"] = (
            last_3.mean()
        )

        future_row["rolling_mean_6"] = (
            last_6.mean()
        )

        future_row["rolling_mean_12"] = (
            last_12.mean()
        )


        # Rolling medians
        future_row["rolling_median_3"] = (
            last_3.median()
        )

        future_row["rolling_median_6"] = (
            last_6.median()
        )

        future_row["rolling_median_12"] = (
            last_12.median()
        )


        # Annual usage
        future_row["rolling_total_12"] = (
            last_12.sum()
        )


        # Standard deviations
        future_row["rolling_std_3"] = (
            last_3.std()
        )

        future_row["rolling_std_6"] = (
            last_6.std()
        )

        future_row["rolling_std_12"] = (
            last_12.std()
        )


        # Usage range
        future_row["rolling_min_12"] = (
            last_12.min()
        )

        future_row["rolling_max_12"] = (
            last_12.max()
        )


        # Zero-month percentage
        future_row[
            "zero_month_percentage_12"
        ] = (
            last_12.eq(0).mean()
        )


        # Coefficient of variation
        rolling_mean_12 = (
            future_row["rolling_mean_12"]
        )

        rolling_std_12 = (
            future_row["rolling_std_12"]
        )

        if rolling_mean_12 != 0:

            future_row[
                "coefficient_variation_12"
            ] = (
                rolling_std_12
                / rolling_mean_12
            )

        else:

            future_row[
                "coefficient_variation_12"
            ] = 0.0


        # Recent versus annual usage
        future_row["recent_vs_annual"] = (
            future_row["rolling_mean_3"]
            -
            future_row["rolling_mean_12"]
        )


        # Usage trends
        future_row["trend_3"] = (
            recent_usage.iloc[-1]
            -
            recent_usage.iloc[-3]
        )

        future_row["trend_6"] = (
            recent_usage.iloc[-1]
            -
            recent_usage.iloc[-6]
        )


        # =====================================================
        # COMMITMENT FEATURES
        # =====================================================

        last_commits_3 = (
            recent_commits.iloc[-3:]
        )

        last_commits_6 = (
            recent_commits.iloc[-6:]
        )

        last_commits_12 = (
            recent_commits.iloc[-12:]
        )


        # Commitment lags
        future_row["commits_lag_1"] = (
            recent_commits.iloc[-1]
        )

        future_row["commits_lag_2"] = (
            recent_commits.iloc[-2]
        )

        future_row["commits_lag_3"] = (
            recent_commits.iloc[-3]
        )

        future_row["commits_lag_6"] = (
            recent_commits.iloc[-6]
        )


        # Rolling commitment averages
        future_row[
            "commits_rolling_mean_3"
        ] = (
            last_commits_3.mean()
        )

        future_row[
            "commits_rolling_mean_6"
        ] = (
            last_commits_6.mean()
        )

        future_row[
            "commits_rolling_mean_12"
        ] = (
            last_commits_12.mean()
        )


        # Rolling commitment totals
        future_row[
            "commits_rolling_sum_3"
        ] = (
            last_commits_3.sum()
        )

        future_row[
            "commits_rolling_sum_6"
        ] = (
            last_commits_6.sum()
        )


        # Recent versus annual commitment level
        future_row[
            "commits_recent_vs_annual"
        ] = (
            future_row[
                "commits_rolling_mean_3"
            ]
            -
            future_row[
                "commits_rolling_mean_12"
            ]
        )


        # =====================================================
        # MODEL INPUT
        # =====================================================

        future_features = pd.DataFrame(
            [future_row],
            columns=feature_columns,
        )


        # =====================================================
        # PREDICT
        # =====================================================

        predicted_usage = float(
            model.predict(
                future_features
            )[0]
        )

        predicted_usage = max(
            0.0,
            predicted_usage,
        )


        predictions.append(
            {
                PART_COLUMN: part_number,
                DATE_COLUMN: next_date,
                "Predicted Usage": predicted_usage,
            }
        )


        # =====================================================
        # UPDATE HISTORY
        # =====================================================

        # The rolling backtest forecasts only one month
        # per call, so this commitment placeholder is
        # not used for the following validation month.
        #
        # Cell 6 rebuilds historical_data and retrieves
        # the real commitment value from the dataset.

        new_history_row = pd.DataFrame(
            {
                DATE_COLUMN: [next_date],
                TARGET_COLUMN: [
                    predicted_usage
                ],
                COMMITS_COLUMN: [0],
            }
        )


        forecast_history = pd.concat(
            [
                forecast_history,
                new_history_row,
            ],
            ignore_index=True,
        )


    return pd.DataFrame(predictions)


print(
    "Version 5 commitment forecast function created."
)

Version 5 commitment forecast function created.


In [0]:
all_results = []
errors = []

validation_start = pd.Timestamp(
    VALIDATION_START
)

validation_months = FORECAST_MONTHS


for part_number in sorted(
    data[PART_COLUMN].dropna().unique()
):

    print(f"\nProcessing: {part_number}")


    try:

        part_data = (
            data[
                data[PART_COLUMN] == part_number
            ]
            .copy()
            .sort_values(DATE_COLUMN)
            .reset_index(drop=True)
        )


        part_results = []


        for month_number in range(
            validation_months
        ):

            prediction_date = (
                validation_start
                + pd.DateOffset(
                    months=month_number
                )
            )


            # Use all actual usage and commitment
            # information available before
            # the prediction month.
            historical_data = part_data[
                part_data[DATE_COLUMN]
                < prediction_date
            ].copy()


            actual_row = part_data[
                part_data[DATE_COLUMN]
                == prediction_date
            ][
                [
                    DATE_COLUMN,
                    TARGET_COLUMN,
                ]
            ].copy()


            if actual_row.empty:

                raise ValueError(
                    f"No actual usage found for "
                    f"{prediction_date:%Y-%m}"
                )


            training_rows, feature_columns = (
                create_training_features(
                    historical_data
                )
            )


            if training_rows.empty:

                raise ValueError(
                    f"No usable training rows for "
                    f"{prediction_date:%Y-%m}"
                )


            # Retrain before each monthly prediction.
            model = train_model(
                training_rows,
                feature_columns,
            )


            # Predict one month ahead.
            one_month_forecast = (
                forecast_future_months(
                    model=model,
                    historical_data=historical_data,
                    feature_columns=feature_columns,
                    forecast_months=1,
                    part_number=part_number,
                )
            )


            predicted_usage = (
                one_month_forecast[
                    "Predicted Usage"
                ].iloc[0]
            )


            actual_usage = (
                actual_row[
                    TARGET_COLUMN
                ].iloc[0]
            )


            error = (
                predicted_usage
                - actual_usage
            )


            part_results.append(
                {
                    PART_COLUMN: part_number,
                    DATE_COLUMN: prediction_date,
                    "Predicted Usage": (
                        predicted_usage
                    ),
                    "Actual Usage": (
                        actual_usage
                    ),
                    "Error": error,
                    "Absolute Error": abs(
                        error
                    ),
                    "Training Through": (
                        historical_data[
                            DATE_COLUMN
                        ].max()
                    ),
                }
            )


        all_results.append(
            pd.DataFrame(
                part_results
            )
        )


        print(
            f"Finished: {part_number}"
        )


    except Exception as error:

        errors.append(
            {
                PART_COLUMN: str(
                    part_number
                ),
                "Error": str(error),
            }
        )

        print(
            f"Skipped {part_number}: "
            f"{error}"
        )


if not all_results:

    raise RuntimeError(
        "No rolling forecasts "
        "completed successfully."
    )


print(
    "\nVersion 5 commitment "
    "rolling backtest complete."
)


Processing: ASY-1118-ENC


Finished: ASY-1118-ENC

Processing: BK15.41-GI-F01-V001


Finished: BK15.41-GI-F01-V001

Processing: CYL-0009


Finished: CYL-0009

Processing: ENG-0124


Finished: ENG-0124

Processing: ENG-0135


...

Finished: KIT-0682

Processing: MTR-0052


Finished: MTR-0052

Processing: SHE-0320


Finished: SHE-0320

Processing: TNK-0099


Finished: TNK-0099

Processing: TNK-0138


Finished: TNK-0138

Version 5 commitment rolling backtest complete.


In [0]:
results = (
    pd.concat(
        all_results,
        ignore_index=True,
    )
    .sort_values(
        [
            PART_COLUMN,
            DATE_COLUMN,
        ]
    )
    .reset_index(drop=True)
)


# Keep original values for accuracy calculations.
# Rounded predictions are for presentation only.
results["Predicted Usage Rounded"] = (
    results["Predicted Usage"]
    .round()
    .astype(int)
)

results["Actual Usage Rounded"] = (
    results["Actual Usage"]
    .round()
    .astype(int)
)


display(
    results[
        [
            PART_COLUMN,
            DATE_COLUMN,
            "Training Through",
            "Predicted Usage Rounded",
            "Actual Usage Rounded",
            "Error",
            "Absolute Error",
        ]
    ]
)

,fpartno,Date,Training Through,Predicted Usage Rounded,Actual Usage Rounded,Error,Absolute Error
0,ASY-1118-ENC,2026-01-01,2025-12-01,2,3,-1.069604,1.069604
1,ASY-1118-ENC,2026-02-01,2026-01-01,3,5,-2.238623,2.238623
2,ASY-1118-ENC,2026-03-01,2026-02-01,2,0,2.385794,2.385794
3,ASY-1118-ENC,2026-04-01,2026-03-01,1,0,0.858485,0.858485
4,ASY-1118-ENC,2026-05-01,2026-04-01,1,2,-1.434316,1.434316
...,...,...,...,...,...,...,...
97,TNK-0138,2026-02-01,2026-01-01,10,5,5.132352,5.132352
98,TNK-0138,2026-03-01,2026-02-01,8,8,0.132935,0.132935
99,TNK-0138,2026-04-01,2026-03-01,3,2,1.301334,1.301334
100,TNK-0138,2026-05-01,2026-04-01,3,6,-3.087629,3.087629


In [0]:
profile_data = data[
    data[DATE_COLUMN]
    < pd.Timestamp(VALIDATION_START)
].copy()


demand_profile = (
    profile_data
    .groupby(PART_COLUMN)
    .agg(

        Average_Monthly_Usage=(
            TARGET_COLUMN,
            "mean",
        ),

        Median_Monthly_Usage=(
            TARGET_COLUMN,
            "median",
        ),

        Zero_Month_Percentage=(
            TARGET_COLUMN,
            lambda x: (
                x == 0
            ).mean(),
        ),

        Nonzero_Months=(
            TARGET_COLUMN,
            lambda x: (
                x > 0
            ).sum(),
        ),

        Average_Commits=(
            COMMITS_COLUMN,
            "mean",
        ),

        Median_Commits=(
            COMMITS_COLUMN,
            "median",
        ),

        Maximum_Commits=(
            COMMITS_COLUMN,
            "max",
        ),

        Months_With_Commits=(
            COMMITS_COLUMN,
            lambda x: (
                x > 0
            ).sum(),
        ),

        Commit_Month_Percentage=(
            COMMITS_COLUMN,
            lambda x: (
                x > 0
            ).mean(),
        ),
    )
    .reset_index()
)


demand_profile[
    "Zero_Month_Percentage"
] *= 100

demand_profile[
    "Commit_Month_Percentage"
] *= 100


# Round for presentation
demand_profile[
    "Average_Monthly_Usage"
] = (
    demand_profile[
        "Average_Monthly_Usage"
    ].round(2)
)

demand_profile[
    "Median_Monthly_Usage"
] = (
    demand_profile[
        "Median_Monthly_Usage"
    ].round(2)
)

demand_profile[
    "Average_Commits"
] = (
    demand_profile[
        "Average_Commits"
    ].round(2)
)

demand_profile[
    "Median_Commits"
] = (
    demand_profile[
        "Median_Commits"
    ].round(2)
)

demand_profile[
    "Zero_Month_Percentage"
] = (
    demand_profile[
        "Zero_Month_Percentage"
    ].round(1)
)

demand_profile[
    "Commit_Month_Percentage"
] = (
    demand_profile[
        "Commit_Month_Percentage"
    ].round(1)
)


display(demand_profile)

,fpartno,Average_Monthly_Usage,Median_Monthly_Usage,Zero_Month_Percentage,Nonzero_Months,Average_Commits,Median_Commits,Maximum_Commits,Months_With_Commits,Commit_Month_Percentage
0,ASY-1118-ENC,1.94,0.0,58.3,15,1.63,0.10,13.333333,18,50.0
1,BK15.41-GI-F01-V001,4.00,2.0,30.6,25,10.51,3.56,40.615385,28,77.8
2,CYL-0009,78.50,75.5,0.0,36,118.56,95.93,212.476191,36,100.0
3,ENG-0124,4.64,4.0,2.8,35,2.69,1.64,16.500000,35,97.2
4,ENG-0135,2.47,1.0,36.1,23,4.06,1.95,23.470588,27,75.0
5,GAG-0006W,88.00,86.0,0.0,36,119.01,87.83,289.970588,36,100.0
6,HOS-0365,0.86,0.5,50.0,18,1.97,1.50,6.000000,34,94.4
7,IDL-0012,63.11,64.5,0.0,36,75.15,61.14,149.250000,36,100.0
8,IK12.14-F08-V001,50.08,48.5,0.0,36,73.99,43.74,193.259259,36,100.0
9,IK120-420-F08-V001,4.50,4.0,5.6,34,4.97,4.25,21.000000,32,88.9


In [0]:
import boto3
from sagemaker_studio import Project
import io


# Get project S3 path
proj = Project()
project_s3_root = proj.s3.root

s3_parts = (
    project_s3_root
    .replace("s3://", "")
    .split("/", 1)
)

bucket = s3_parts[0]

prefix = (
    s3_parts[1]
    if len(s3_parts) > 1
    else ""
)

s3 = boto3.client("s3")


# Select results to upload
results_to_upload = results[
    [
        PART_COLUMN,
        DATE_COLUMN,
        "Training Through",
        "Predicted Usage",
        "Predicted Usage Rounded",
        "Actual Usage",
        "Error",
        "Absolute Error",
    ]
].copy()


# Create CSV in memory
csv_buffer = io.StringIO()

results_to_upload.to_csv(
    csv_buffer,
    index=False,
)


# Separate Version 5 output file
s3_key = (
    f"{prefix}/results/"
    "version_5_commitments_rolling_backtest_results.csv"
)


# Upload to S3
s3.put_object(
    Bucket=bucket,
    Key=s3_key,
    Body=csv_buffer.getvalue().encode(
        "utf-8"
    ),
    ContentType="text/csv",
)


s3_path = (
    f"s3://{bucket}/{s3_key}"
)


print(
    "Successfully uploaded Version 5 "
    "commitment rolling backtest results to S3!"
)

print(
    f"S3 path: {s3_path}"
)

print(
    f"Rows uploaded: "
    f"{len(results_to_upload)}"
)

Successfully uploaded Version 5 July-December 2026 forecast to S3!
S3 path: s3://amazon-sagemaker-369282953854-us-east-1-7ffd8a98b34e/dzd-bokklg2avhnkh3/6vqwkr3e84m5d3/dev/results/version_5_july_december_2026_forecast.csv
Rows uploaded: 102
